In [ ]:
import deepfmkit.core as dfm
from deepfmkit.plotting import default_rc, apply_legend
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update(default_rc)

# Instantiate the main framework
dff = dfm.DeepFrame()

# --- 1. Define the Single, Shared Laser Source ---
laser_config = dfm.LaserConfig(label="main_laser")
laser_config.fm = 1000  # Modulation frequency (Hz)
# Make laser frequency noise significant to demonstrate tracking
laser_config.f_n = 100e6  # Laser frequency noise at 1 Hz (Hz/rtHz)

# --- 2. Define the Main Interferometer ---
main_ifo_config = dfm.IfoConfig(label="dynamic_ifo")
main_ifo_config.ref_arml = 0.1  # Reference arm length (m)
main_ifo_config.meas_arml = 0.3  # Measurement arm length (m)

# --- 3. Set Modulation Depth by Adjusting Laser's `df` ---
m_target = 6.0  # Target effective modulation index (rad)
laser_config.set_df_for_m(main_ifo_config, m_target)

# --- 4. Compose the Main Channel ---
main_label = "dynamic_channel"
main_channel = dfm.SimConfig(
    label=main_label,
    laser_config=laser_config,
    ifo_config=main_ifo_config,
    f_samp=int(200e3),  # Sampling frequency (Hz)
)
dff.add_sim(main_channel)

# --- 5. Simulate ---
dff.simulate(
    label=main_label,
    n_seconds=5,  # Simulation length in seconds
    verbose=True,
)

# --- 6. Fit ---

# NLS fit
print("\n--- Running NLS Fit ---")
dff.fit(main_label, fit_label="nls", method="nls", n_harmonics=30, n_cycles=100)

# IntegratedEKF with default tuning parameters
print("\n--- Running Integrated EKF (Default) ---")
dff.fit(main_label, fit_label="iekf (default)", method="iekf", n_cycles=1)

# --- 7. Tuning the IntegratedEKF ---
print("\n--- Running Integrated EKF (Tuned) ---")

# The state is [phi, phi_dot, psi, psi_dot, m, m_dot, c, c_dot, a, a_dot]
# Decrease the process noise by a lot (more sluggish).
tuned_q_diag = [1e-8] * 10

# Create a dictionary of the parameters we want to pass to the fitter's constructor.
# This dictionary will be passed directly to the `fit_config` argument.
tuned_fitter_config = {
    'Q_diag': tuned_q_diag,
    'P0_diag': [0.1] * 10
}

dff.fit(
    main_label,
    fit_label="iekf (tuned)",
    method="iekf", n_cycles=1,
    fit_config=tuned_fitter_config # Pass the dictionary directly
)

print("\nPlotting results...")

r_nls = dff.fits["nls"]
r_iekf = dff.fits["iekf (default)"]
r_iekf_t = dff.fits["iekf (tuned)"]
f_samp = dff.raws['dynamic_channel'].f_samp
gt = dff.raws['dynamic_channel'].phi_sim_noisy
time = np.arange(len(gt))/f_samp

r_nls_time_0 = r_nls.time 
r_nls_time_1 = r_nls_time_0 + (r_nls_time_0[1] - r_nls_time_0[0])
r_nls_time_1 = r_nls_time_0

fig, ax = plt.subplots(figsize=(7, 2.3), dpi=150)

ax.plot(time, gt, linestyle='-', linewidth=1.5, color='gray', label="Groundtruth")

ax.plot(r_iekf.time, r_iekf.phi, c='tomato', label="IntegratedEKF (responsive)")
ax.plot(r_iekf_t.time, r_iekf_t.phi, c='deepskyblue', label="IntegratedEKF (sluggish)")

ax.scatter(r_nls_time_1, r_nls.phi, s=6, marker='o', color='k', label="StandardNLS", zorder=10)
ax.plot(r_nls_time_1, r_nls.phi, linestyle='--', linewidth=1.0, color='k', zorder=10)

ax.set_ylabel(r"$\Phi$ (rad)")
ax.set_xlabel("Time (s)")
ax.legend(handlelength=1.9, fontsize=7, edgecolor='k', framealpha=0)
ax.set_ylim(-2.5,2.0)

fig.tight_layout()
plt.show()

fig.savefig("fig_readout.pdf")
